## تثبيت المكتبات

In [1]:
! pip install unsloth
! pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 122.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 37.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [unsloth]


## الاستيراد والإعدادات

In [3]:
from os.path import join
import json
import random
from tqdm.auto import tqdm
from datasets import load_dataset
import torch
model_id = 'Qwen/Qwen2.5-2.5B-Instruct'
device = 'cuda'

## تسجيل الدخول إلى Hugging Face

In [4]:
from unsloth import FastLanguageModel
import os
import torch
from google.colab import userdata
os.environ['HF_token']=userdata.get('HF_TOKEN')
from huggingface_hub import login
login()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1122 20:36:20.531000 3216 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 20:36:20.598000 3216 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


## تحميل النموذج

In [10]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## إعداد قالب المحادثة

In [11]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'qwen-2.5'
)

## تحميل البيانات

In [30]:
train_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split = "train",
)
text_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split = "test",)

print(train_dataset)
print(train_dataset[0])
print(len(train_dataset))

ValueError: Unknown split "test". Should be one of ['train'].

## دالة تنسيق البيانات

In [31]:
def formatting_func(example):

    messages = [
        {
        "role": "system",
        "content": "You are a professional customer service assistant. Be polite, helpful, and concise."
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

## تطبيق التنسيق على البيانات

In [32]:
formatted_train_dataset =train_dataset.map(
    formatting_func,

)

## عرض مثال بعد التنسيق

In [34]:
print(formatted_train_dataset[0])

{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.", 'text': "<|im_start|>system\nYou are a professional customer service assistant. Be polite, helpful, and concise.<|im_end|>\n<|im_start|>user\nquestion about cancelling order {{Order Number}}<|im_end|>\n<|im_start|>assistant\nI've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.<|im_end|>\n"}


## تجهيز رسالة الاختبار

In [35]:
messages = [
       {
        "role": "system",
        "content": "You are a professional customer service assistant. Be polite, helpful, and concise."
        },
        {
            "role": "user",
            "content": "question about cancelling order"
        }
       ]

## اختبار النموذج قبل التدريب

In [36]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True

)
print(text)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(device)

output = model.generate(
    inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False,
)

input_length = len(inputs.input_ids[0])

generated_ids = output[0][input_length:]
generated_text = tokenizer.decode(generated_ids)
print(generated_text)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are a professional customer service assistant. Be polite, helpful, and concise.<|im_end|>
<|im_start|>user
question about cancelling order<|im_end|>
<|im_start|>assistant

Hello! I'm here to help with any questions you might have about canceling an order. Could you please provide me with more details? What specific information do you need from the order or what is your question regarding cancellation?<|im_end|>


## إعداد LoRA و Trainer

In [38]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth 2026.11.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/26872 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## بدء التدريب

In [39]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,872 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
1,2.092726
2,1.836635
3,1.904379
4,2.046817
5,1.803237
6,1.726408
7,1.622882
8,1.505845
9,1.350583
10,1.250910


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


## اختبار النموذج بعد التدريب

In [40]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True

)
print(text)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(device)

output = trainer.model.generate(
    inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False,
)

input_length = len(inputs.input_ids[0])

generated_ids = output[0][input_length:]
generated_text = tokenizer.decode(generated_ids)
print(generated_text)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are a professional customer service assistant. Be polite, helpful, and concise.<|im_end|>
<|im_start|>user
question about cancelling order<|im_end|>
<|im_start|>assistant

I'm here to assist you with your inquiry regarding the cancellation of an order. To cancel an order, please visit our website or contact us at {{Customer Support Phone Number}}. Our dedicated team will be more than happy to guide you through the process and ensure that your order is cancelled promptly. Rest assured, we value your satisfaction and appreciate any feedback you may have.<|im_end|>
